# Phase 2C — Per-Class Threshold Tuning

**What this notebook does:**
Replaces the default argmax decision rule with per-class probability thresholds tuned on the Phase 1 OOF predictions. No retraining — the model weights never change.

**Why this is the right first step:**
Phase 1 showed the LP sits at ~0.83 AUROC, but R1 sensitivity is 0.406, R2 is 0.527, R3A is 0.517.
AUROC measures the model's *ranking ability* — it is already good.
Sensitivity measures the *decision boundary* — that is entirely under our control via thresholds.
Threshold tuning converts ranking ability into sensitivity gains for free, before touching the model.

**What we use:**
- `oof_labels_all.npy` / `oof_probs_all.npy` from Phase 1 (4075 images, 990 patients) — to find thresholds
- `test_ensemble_probs.npy` / `test_ensemble_labels.npy` — to evaluate final performance

In [ ]:
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, accuracy_score, cohen_kappa_score,
    confusion_matrix, roc_curve
)

os.chdir(os.path.dirname(os.path.abspath('phase2c_threshold_tuning.ipynb')))

CLASSES      = ['R0', 'R1', 'R2', 'R3A']
NUM_CLASSES  = 4
CV_OUTPUT    = Path('output_dir/phase1_cv')
THRESH_OUTPUT = Path('output_dir/phase2c_thresholds')
THRESH_OUTPUT.mkdir(parents=True, exist_ok=True)

# ── Load OOF predictions (used to FIND thresholds) ────────────────────────────
oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')
oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy').astype(np.float64)
oof_probs  = oof_probs / oof_probs.sum(axis=1, keepdims=True)

# ── Load test ensemble predictions (used to EVALUATE thresholds) ──────────────
test_labels = np.load(CV_OUTPUT / 'test_ensemble_labels.npy')
test_probs  = np.load(CV_OUTPUT / 'test_ensemble_probs.npy')
# test_ensemble_probs was already saved as float64 and normalised in phase1 notebook
test_probs  = test_probs / test_probs.sum(axis=1, keepdims=True)

print(f'OOF  : {len(oof_labels)} images, label distribution: {dict(zip(*np.unique(oof_labels, return_counts=True)))}')
print(f'Test : {len(test_labels)} images, label distribution: {dict(zip(*np.unique(test_labels, return_counts=True)))}')

## Helper: metrics from predicted labels

We separate metric computation from threshold application so we can reuse it cleanly.

In [ ]:
def compute_metrics(labels, probs, preds=None):
    """Compute full metrics. preds defaults to argmax if not supplied."""
    if preds is None:
        preds = probs.argmax(axis=1)
    probs_f64 = probs.astype(np.float64)
    probs_f64 = probs_f64 / probs_f64.sum(axis=1, keepdims=True)
    try:
        auroc = roc_auc_score(labels, probs_f64, multi_class='ovr', average='macro',
                              labels=list(range(NUM_CLASSES)))
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    cm    = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else float('nan'))
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))
    return {
        'auroc': auroc, 'accuracy': acc, 'kappa': kappa,
        'macro_sensitivity': float(np.nanmean(sens)),
        'macro_specificity': float(np.nanmean(spec)),
        'sensitivity': np.array(sens),
        'specificity': np.array(spec),
        'cm': cm,
    }


def apply_thresholds(probs, thresholds):
    """
    OvR threshold decision rule:
      For each sample, find which classes have p_i > threshold_i.
      If exactly one: predict that class.
      If multiple: predict the one with highest p_i / threshold_i (most confident relative to its bar).
      If none: fall back to argmax (the model's best guess with no class clearing its threshold).
    """
    thresholds = np.array(thresholds)
    # Ratio of probability to threshold — rescales each class by its bar
    ratios = probs / thresholds
    above  = probs > thresholds          # bool mask (n, 4)
    preds  = np.where(above.any(axis=1),
                      ratios.argmax(axis=1),   # at least one class above threshold
                      probs.argmax(axis=1))    # fallback: argmax
    return preds


# ── Argmax baseline (what Phase 1 used) ───────────────────────────────────────
baseline_oof  = compute_metrics(oof_labels,  oof_probs)
baseline_test = compute_metrics(test_labels, test_probs)

print('ARGMAX BASELINE — OOF')
print(f'  AUROC={baseline_oof["auroc"]:.4f}  kappa={baseline_oof["kappa"]:.4f}  '
      f'macro_sens={baseline_oof["macro_sensitivity"]:.4f}')
for i, c in enumerate(CLASSES):
    print(f'  {c}: sens={baseline_oof["sensitivity"][i]:.4f}  spec={baseline_oof["specificity"][i]:.4f}')
print()
print('ARGMAX BASELINE — TEST ENSEMBLE')
print(f'  AUROC={baseline_test["auroc"]:.4f}  kappa={baseline_test["kappa"]:.4f}  '
      f'macro_sens={baseline_test["macro_sensitivity"]:.4f}')
for i, c in enumerate(CLASSES):
    print(f'  {c}: sens={baseline_test["sensitivity"][i]:.4f}  spec={baseline_test["specificity"][i]:.4f}')

## Step 1 — OvR ROC curves and Youden's J thresholds

For each class `i` independently, we create a binary problem:
- Positive label: image is class `i`
- Score: `p_i` (the model's probability for class `i`)

The ROC curve plots sensitivity (true positive rate) vs 1−specificity (false positive rate) as we slide the threshold from 1.0 down to 0.0.

**Youden's J statistic** = sensitivity + specificity − 1. It peaks at the threshold that best balances catching true cases (sensitivity) while avoiding false alarms (specificity). This is a neutral starting point — not biased toward either.

We compute this for all four classes, but only R1, R2, R3A have sensitivity problems. R0 at 0.852 is fine as-is.

In [ ]:
# ── OvR ROC curves ────────────────────────────────────────────────────────────
roc_data   = {}   # class -> (fpr, tpr, thresholds)
youden_thr = []   # one threshold per class

print('OvR ROC analysis (on OOF predictions):')
print(f'{"Class":<6} {"AUROC":>7} {"Youden_thr":>11} {"sens@Youden":>12} {"spec@Youden":>12}')
print('-' * 52)

for i, c in enumerate(CLASSES):
    binary_labels = (oof_labels == i).astype(int)
    scores        = oof_probs[:, i]

    fpr, tpr, thrs = roc_curve(binary_labels, scores)
    auc_i = roc_auc_score(binary_labels, scores)

    # Youden's J = tpr + (1-fpr) - 1 = tpr - fpr
    j        = tpr - fpr
    best_idx = j.argmax()
    best_thr = float(thrs[best_idx])
    best_sen = float(tpr[best_idx])
    best_spe = float(1 - fpr[best_idx])

    roc_data[c]   = (fpr, tpr, thrs)
    youden_thr.append(best_thr)

    print(f'{c:<6} {auc_i:>7.4f} {best_thr:>11.4f} {best_sen:>12.4f} {best_spe:>12.4f}')

print()
print(f'Youden thresholds: {[f"{t:.4f}" for t in youden_thr]}')
print('(Default argmax is equivalent to threshold=0.25 for a uniform 4-class model;')
print(' Youden thresholds below 0.25 lower the bar for that class → higher sensitivity)')

## Step 2 — Apply Youden thresholds and measure the gain

Now we apply the Youden thresholds to both the OOF pool and the test ensemble, and compare to the argmax baseline.

**Important:** we optimised thresholds on OOF data, so OOF improvement is expected. The test-ensemble number is the honest evaluation — thresholds have never seen those 702 images.

In [ ]:
preds_youden_oof  = apply_thresholds(oof_probs,  youden_thr)
preds_youden_test = apply_thresholds(test_probs, youden_thr)

m_youden_oof  = compute_metrics(oof_labels,  oof_probs,  preds_youden_oof)
m_youden_test = compute_metrics(test_labels, test_probs, preds_youden_test)

def print_comparison(label_a, m_a, label_b, m_b, on='OOF'):
    print(f'\n{"=" * 64}')
    print(f'  {on}: {label_a}  vs  {label_b}')
    print(f'{"=" * 64}')
    print(f'{"Metric":<22} {label_a:>14} {label_b:>14} {"Δ":>8}')
    print('-' * 60)
    for key, name in [("auroc","AUROC"), ("kappa","Kappa"), ("accuracy","Accuracy"),
                      ("macro_sensitivity","Macro sens."), ("macro_specificity","Macro spec.")]:
        va, vb = m_a[key], m_b[key]
        print(f'{name:<22} {va:>14.4f} {vb:>14.4f} {vb-va:>+8.4f}')
    print()
    print(f'{"Class":<6} {"sens_"+label_a[:4]:>10} {"sens_"+label_b[:4]:>10} {"Δsens":>8}  '
          f'{"spec_"+label_a[:4]:>10} {"spec_"+label_b[:4]:>10} {"Δspec":>8}')
    print('-' * 72)
    for i, c in enumerate(CLASSES):
        sa, sb = m_a['sensitivity'][i], m_b['sensitivity'][i]
        pa, pb = m_a['specificity'][i], m_b['specificity'][i]
        print(f'{c:<6} {sa:>10.4f} {sb:>10.4f} {sb-sa:>+8.4f}  {pa:>10.4f} {pb:>10.4f} {pb-pa:>+8.4f}')

print_comparison('Argmax', baseline_oof,  'Youden', m_youden_oof,  on='OOF')
print_comparison('Argmax', baseline_test, 'Youden', m_youden_test, on='TEST')

## Step 3 — Fixed sensitivity targets

Youden's J is a neutral balance point. Clinically, you may want to guarantee a minimum sensitivity — for example: *"catch at least 70% of R2 and R3A patients, accept lower specificity if needed."*

For each class, we scan the ROC curve and find the **highest threshold** that still achieves the target sensitivity. Higher threshold = fewer false positives = better specificity, so we want the largest threshold that meets the target.

We show three target levels: 0.65, 0.70, 0.75 for R2 and R3A. R0 and R1 thresholds are kept at their Youden values (they're not the bottleneck).

In [ ]:
def find_threshold_for_sensitivity(fpr, tpr, thrs, target_sens):
    """
    Return the highest threshold where TPR >= target_sens.
    Higher threshold = higher specificity = fewer false positives.
    Returns None if target is unachievable.
    """
    # roc_curve returns thrs in decreasing order; tpr increases as thrs decreases
    for thr, tp in zip(thrs, tpr):
        if tp >= target_sens:
            return float(thr), float(tp), float(1 - fpr[list(tpr).index(tp)])
    return None, None, None


targets = [0.65, 0.70, 0.75]

print('Fixed-sensitivity threshold search (R2 and R3A):')
print(f'{"Target sens":>12}  {"Class":>5}  {"Threshold":>10}  {"Achieved sens":>14}  {"Spec at thr":>12}')
print('-' * 60)

target_thresholds = {}   # target -> [t0, t1, t2, t3]

for target in targets:
    # R0 and R1: keep Youden thresholds (they're not the problem classes)
    thr_row = [youden_thr[0], youden_thr[1], None, None]
    for i, c in enumerate(['R2', 'R3A']):
        ci = i + 2   # R2=2, R3A=3
        fpr, tpr, thrs = roc_data[c]
        thr, achieved_sens, achieved_spec = find_threshold_for_sensitivity(
            fpr, tpr, thrs, target
        )
        thr_row[ci] = thr
        if thr is not None:
            print(f'{target:>12.2f}  {c:>5}  {thr:>10.4f}  {achieved_sens:>14.4f}  {achieved_spec:>12.4f}')
        else:
            print(f'{target:>12.2f}  {c:>5}  {"N/A":>10}  {"unachievable":>14}')
    target_thresholds[target] = thr_row

print()
print('Note: thresholds for R0 and R1 are held at their Youden values.')
print('Lowering the threshold for a class increases its sensitivity but lowers its specificity.')

## Step 4 — Evaluate each target operating point

Now we apply each set of thresholds to both OOF and test, and build a summary table.
This lets you pick the operating point that fits your clinical requirement.

In [ ]:
rows = []

# Baseline (argmax)
rows.append({'name': 'Argmax (baseline)',
             'thresholds': [0.25]*4,
             'oof':  baseline_oof,
             'test': baseline_test})

# Youden
rows.append({'name': 'Youden J (balanced)',
             'thresholds': youden_thr,
             'oof':  m_youden_oof,
             'test': m_youden_test})

# Fixed-sensitivity targets
for target in targets:
    thrs = target_thresholds[target]
    if any(t is None for t in thrs):
        continue
    m_oof_t  = compute_metrics(oof_labels,  oof_probs,  apply_thresholds(oof_probs,  thrs))
    m_test_t = compute_metrics(test_labels, test_probs, apply_thresholds(test_probs, thrs))
    rows.append({'name': f'Target sens≥{target:.2f}',
                 'thresholds': thrs,
                 'oof':  m_oof_t,
                 'test': m_test_t})

# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 100)
print('THRESHOLD OPERATING POINTS — OOF')
print('=' * 100)
hdr = f'{"Operating point":<22} {"AUROC":>6} {"Kappa":>6} {"MacSens":>8} {"R0_s":>6} {"R1_s":>6} {"R2_s":>6} {"R3A_s":>7} {"R0_p":>6} {"R1_p":>6} {"R2_p":>6} {"R3A_p":>7}'
print(hdr)
print('-' * 100)
for r in rows:
    m = r['oof']
    s = m['sensitivity']; p = m['specificity']
    print(f"{r['name']:<22} {m['auroc']:>6.4f} {m['kappa']:>6.4f} {m['macro_sensitivity']:>8.4f} "
          f"{s[0]:>6.4f} {s[1]:>6.4f} {s[2]:>6.4f} {s[3]:>7.4f} "
          f"{p[0]:>6.4f} {p[1]:>6.4f} {p[2]:>6.4f} {p[3]:>7.4f}")

print()
print('=' * 100)
print('THRESHOLD OPERATING POINTS — TEST ENSEMBLE (honest evaluation, never seen during threshold search)')
print('=' * 100)
print(hdr)
print('-' * 100)
for r in rows:
    m = r['test']
    s = m['sensitivity']; p = m['specificity']
    print(f"{r['name']:<22} {m['auroc']:>6.4f} {m['kappa']:>6.4f} {m['macro_sensitivity']:>8.4f} "
          f"{s[0]:>6.4f} {s[1]:>6.4f} {s[2]:>6.4f} {s[3]:>7.4f} "
          f"{p[0]:>6.4f} {p[1]:>6.4f} {p[2]:>6.4f} {p[3]:>7.4f}")

## Step 5 — Sensitivity / specificity tradeoff curves

The table above shows discrete operating points. This cell prints the full tradeoff curve for R2 and R3A so you can see exactly what you give up in specificity for each gain in sensitivity.
This is the key clinical decision: pick where on this curve you want to operate.

In [ ]:
print('Sensitivity / Specificity tradeoff curves (OvR on OOF pool)')
print('Read: if you set the threshold to the value in column 3, you get that sensitivity and specificity.')
print()

for c in ['R2', 'R3A']:
    i   = CLASSES.index(c)
    fpr, tpr, thrs = roc_data[c]
    print(f'--- {c} ---')
    print(f'  {"Threshold":>10}  {"Sensitivity":>12}  {"Specificity":>12}  {"Youden J":>10}')
    # Print ~20 evenly-spaced points along the curve
    step = max(1, len(thrs) // 20)
    prev_sen = -1
    for idx in range(0, len(thrs), step):
        sen = tpr[idx]
        spe = 1 - fpr[idx]
        thr = thrs[idx]
        j   = sen + spe - 1
        if abs(sen - prev_sen) < 0.01:   # skip near-duplicates
            continue
        prev_sen = sen
        marker = ' <-- Youden J' if abs(thr - youden_thr[CLASSES.index(c)]) < 0.005 else ''
        print(f'  {thr:>10.4f}  {sen:>12.4f}  {spe:>12.4f}  {j:>10.4f}{marker}')
    print()

## Step 6 — Save chosen thresholds

Pick the operating point you want and set `CHOSEN_TARGET` below.
The thresholds are saved to `output_dir/phase2c_thresholds/thresholds.json`.

Options:
- `'youden'` — balanced, maximises Youden J for each class
- `0.65` — R2 and R3A sensitivity ≥ 0.65
- `0.70` — R2 and R3A sensitivity ≥ 0.70
- `0.75` — R2 and R3A sensitivity ≥ 0.75 (may be unachievable)

In [ ]:
CHOSEN_TARGET = 'youden'   # ← change this to 0.65 / 0.70 / 0.75 if you prefer

if CHOSEN_TARGET == 'youden':
    chosen = [r for r in rows if r['name'] == 'Youden J (balanced)'][0]
else:
    chosen = [r for r in rows if r['name'] == f'Target sens≥{CHOSEN_TARGET:.2f}'][0]

chosen_thresholds = chosen['thresholds']
chosen_test       = chosen['test']

print(f'Chosen operating point: {chosen["name"]}')
print(f'Thresholds: {[f"{t:.4f}" for t in chosen_thresholds]}')
print()
print('TEST ENSEMBLE results with chosen thresholds:')
print(f'  AUROC       : {chosen_test["auroc"]:.4f}')
print(f'  Kappa       : {chosen_test["kappa"]:.4f}')
print(f'  Macro sens  : {chosen_test["macro_sensitivity"]:.4f}')
print(f'  Macro spec  : {chosen_test["macro_specificity"]:.4f}')
print()
print(f'{"Class":<6} {"Sensitivity":>12} {"Specificity":>12}  (vs argmax: sens  spec)')
for i, c in enumerate(CLASSES):
    s = chosen_test['sensitivity'][i]
    p = chosen_test['specificity'][i]
    bs = baseline_test['sensitivity'][i]
    bp = baseline_test['specificity'][i]
    print(f'{c:<6} {s:>12.4f} {p:>12.4f}  (Δsens={s-bs:+.4f}  Δspec={p-bp:+.4f})')

# ── Save ──────────────────────────────────────────────────────────────────────
save_dict = {
    'operating_point': str(CHOSEN_TARGET),
    'thresholds': {c: float(chosen_thresholds[i]) for i, c in enumerate(CLASSES)},
    'youden_thresholds': {c: float(youden_thr[i]) for i, c in enumerate(CLASSES)},
    'all_target_thresholds': {
        str(t): {c: float(target_thresholds[t][i]) for i, c in enumerate(CLASSES)}
        for t in targets if all(v is not None for v in target_thresholds[t])
    },
    'test_results': {
        'argmax': {
            'auroc': float(baseline_test['auroc']),
            'kappa': float(baseline_test['kappa']),
            'macro_sensitivity': float(baseline_test['macro_sensitivity']),
            'sensitivity': baseline_test['sensitivity'].tolist(),
            'specificity': baseline_test['specificity'].tolist(),
        },
        'chosen': {
            'auroc': float(chosen_test['auroc']),
            'kappa': float(chosen_test['kappa']),
            'macro_sensitivity': float(chosen_test['macro_sensitivity']),
            'sensitivity': chosen_test['sensitivity'].tolist(),
            'specificity': chosen_test['specificity'].tolist(),
        },
    },
}
with open(THRESH_OUTPUT / 'thresholds.json', 'w') as f:
    json.dump(save_dict, f, indent=2)
print(f'\nSaved to {THRESH_OUTPUT}/thresholds.json')